# Generate candidates on a free Colab GPU

This notebook runs `scripts/generate_candidates.py` with a local Hugging Face model
(`HFLocalProvider`, default **Qwen/Qwen2.5-3B-Instruct**, ungated) instead of Ollama.
It writes the same `data/candidates/<TICKER>.jsonl` format as a run on the Mac, so the
files can be committed and labeled the same way. Nothing here calls a paid API.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The repo's
`data/chunks/*.jsonl` and question files must be committed and pushed, because the
notebook clones the repo and builds the FAISS indexes from the committed chunks.

## 1. Clone and install

For a private repo, add a `GITHUB_TOKEN` Colab secret (key icon on the left) with read access.

In [ ]:
import os
import subprocess

REPO = "krupal2241/GroundCheck"
BRANCH = "main"
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.isdir("GroundCheck"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, "GroundCheck"], check=True
    )
%cd GroundCheck
!pip install -q -e ".[embed,hf]"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls data/chunks data/questions data/eval

## 2. Settings

Change the model or the companies here. `MODEL` can be any ungated instruct model that fits the GPU.

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
COMPANIES = ["AAPL", "MSFT", "AMZN", "XOM", "JNJ", "WMT"]  # labeled companies only
BATCH_SIZE = 10

## 3. Build the per-company indexes (bge-small on the GPU)

In [ ]:
!groundcheck index --only {" ".join(COMPANIES)} --device cuda

## 4. Rehearse with the fake provider (no model, output under data/cache/dry-run/)

In [ ]:
!python scripts/generate_candidates.py --dry-run --limit 2 --device cuda \
    --company {" ".join(COMPANIES)}

## 5. Check the model loads and answers

In [ ]:
!groundcheck llm-info --provider hf-local --model {MODEL}

## 6. Generate

Resumable: if the runtime disconnects, re-run this cell (after steps 1 and 3) and it
skips every question that already has a record. Responses are also cached under
`data/cache/llm/`, which lives only as long as the runtime.

In [ ]:
!python scripts/generate_candidates.py --provider hf-local --model {MODEL} \
    --company {" ".join(COMPANIES)} --batch-size {BATCH_SIZE} --device cuda

In [ ]:
!wc -l data/candidates/*.jsonl
!head -c 1500 data/candidates/{COMPANIES[0]}.jsonl

## 7. Download the candidates

Unzip into the repo on your Mac (`data/candidates/`), check `git diff`, and commit.

In [ ]:
!zip -q -r candidates.zip data/candidates
from google.colab import files

files.download("candidates.zip")